# ACLs,Permissions · Secrets & Service Principals · Delta Sharing (D2D/D2O) · Lakehouse Federation

Companion to `ACLs_Compute_Sharing_Federation.pptx`. Continues the QuickBite QSR
domain from Part 1.

**Every section is tagged** so it's always clear what actually executes on Databricks
Free Edition today vs. what's real, correct reference syntax you'd run with more
headroom (a second workspace, account-console access, or an external database):

- 🟢 **RUNS TODAY** — executes as-is, on this exact account
- 🟡 **PARTIAL** — runs partway, or needs something external you'd set up (a second
  workspace, a reachable database)
- 🔴 **CONCEPTUAL ONLY** — genuine, correct syntax, but needs account-console access
  or classic compute, neither of which exist on Free Edition

Confirmed while building this: Free Edition has **one workspace and one metastore per
account**, **no account console or account-level API access**, and **serverless-only
compute**. Nothing below works around those — they're by design, not a setup mistake.

## Setup

In [0]:
dbutils.widgets.text("catalog", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema", "uc_governance_demo", "Schema (reuses Part 1's, if present)")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

from pyspark.sql import functions as F

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

CUSTOMERS = f"{catalog}.{schema}.customers"
current_user = spark.sql("SELECT current_user() AS u").collect()[0].u
print(f"catalog.schema = {catalog}.{schema}")
print(f"current_user() = {current_user}")

# Rebuild a small customers table in case Part 1's isn't present in this session
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CUSTOMERS} (
  customer_id INT, full_name STRING, email STRING, ssn STRING, city STRING, region STRING
)
USING DELTA
""")
if spark.table(CUSTOMERS).count() == 0:
    (
        spark.createDataFrame(
            [(1, "Alice Rao", "alice@example.com", "123-45-6789", "Bengaluru", "South"),
             (2, "Ben Fernandes", "ben@example.com", "987-65-4321", "Mumbai", "West")],
            ["customer_id", "full_name", "email", "ssn", "city", "region"],
        ).write.format("delta").mode("overwrite").saveAsTable(CUSTOMERS)
    )
print(f"{CUSTOMERS}: {spark.table(CUSTOMERS).count()} rows")

catalog.schema = main.uc_governance_demo
current_user() = reshma@platformatory.com
main.uc_governance_demo.customers: 5 rows


## Section 1 — ACLs & Permission Inheritance 🟢 RUNS TODAY

Everything in this section genuinely executes solo — creating grants, and reading
them back via `SHOW GRANTS`, doesn't require a second real identity. Only the very
last step (actually logging in as someone else to test denial/access) needs a
second user, and that part is clearly marked.

In [0]:
# Grant at the catalog level — this one statement covers every schema and table under
# it, including ones that don't exist yet.
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `account users`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `account users`")
spark.sql(f"GRANT SELECT ON TABLE {CUSTOMERS} TO `account users`")

display(spark.sql(f"SHOW GRANTS ON TABLE {CUSTOMERS}"))
display(spark.sql(f"SHOW GRANTS ON SCHEMA {catalog}.{schema}"))
display(spark.sql(f"SHOW GRANTS ON CATALOG {catalog}"))

print("\nNotice SELECT on the TABLE is explicit, but USE CATALOG/USE SCHEMA at higher")
print("levels is what makes that SELECT actually reachable — without those, SELECT alone")
print("wouldn't let anyone navigate down to the table in the first place.")

### 🖱️ UI Instructions: Managing Permissions in Catalog Explorer

**To grant permissions via the UI (instead of SQL):**

1. **Navigate to Catalog Explorer:**
   * Click the left sidebar **Catalog** icon
   * Browse to your catalog → schema → table

2. **Grant permissions on a table:**
   * Select the table (`main.uc_governance_demo.customers`)
   * Click the **Permissions** tab
   * Click **Grant** button
   * Select **Principal** (user, group, or service principal)
   * Check the permissions to grant: `SELECT`, `MODIFY`, `READ_METADATA`
   * Click **Grant**

3. **Grant permissions on schema/catalog:**
   * Same process, but select the schema or catalog level
   * Available permissions: `USE SCHEMA`, `USE CATALOG`, `CREATE TABLE`, etc.

4. **View existing grants:**
   * Click **Permissions** tab on any UC object
   * Shows all principals and their privileges
   * Can revoke by clicking the **X** next to a grant

**Permission inheritance:** Grants at higher levels (catalog → schema → table) flow down automatically.

### 1.1 — If you had a second real user (the exact code to run)

This is genuine, correct syntax — written as though `priya@quickbite.com` is a real
second user in your workspace. On Free Edition, adding a second user to the same
workspace isn't available, so this cell is reference, not executable, today.

```sql
-- Grant a specific person SELECT only (narrower than the account-wide grant above)
GRANT USE CATALOG ON CATALOG quickbite_prod TO `priya@quickbite.com`;
GRANT USE SCHEMA ON SCHEMA quickbite_prod.gold TO `priya@quickbite.com`;
GRANT SELECT ON TABLE quickbite_prod.gold.customers TO `priya@quickbite.com`;

-- Confirm what she can see, from YOUR session (no login needed for this part):
SHOW GRANTS TO `priya@quickbite.com`;

-- The part that genuinely needs her to log in herself:
-- as priya@quickbite.com, in a notebook of her own:
SELECT * FROM quickbite_prod.gold.customers;        -- succeeds (SELECT granted)
INSERT INTO quickbite_prod.gold.customers VALUES (...);  -- fails (MODIFY not granted)
```
**What you'd expect to see:** the `SELECT` succeeds and returns the same rows you see
as the owner. The `INSERT` fails with a permission-denied error naming the exact
missing privilege (`MODIFY`) — this is the single clearest way to demonstrate that
privileges are additive and specific, not all-or-nothing.

### 👥 How to Add a Second User (Another Free Edition Account) -- not applicable

**On Databricks Free Edition**, you cannot add multiple users to a single workspace. However, you can simulate a second-user scenario:

**Option 1: Create a second Free Edition account (Recommended for testing)**
1. Sign up for another Databricks Community Edition account with a different email
2. Have the first account create a **Delta Share** of the table
3. The second account can access the shared data as a recipient

**Option 2: Test with a colleague's Free Edition account**
1. Partner with someone who has their own Free Edition workspace
2. Share data via Delta Sharing (see Section 8 below)
3. This demonstrates real cross-account access control

**Option 3: Upgrade to Databricks Trial (14-day free trial)**
* Go to databricks.com and start a trial
* Add multiple users via: **Settings** → **Identity and Access** → **Users** → **Add User**
* Enter their email address
* They'll receive an invitation to join your workspace
* Then you can grant specific permissions to that user

**To test permissions as the second user:**
1. Log in as that user in a separate browser (or incognito window)
2. Navigate to Catalog Explorer → your catalog → schema → table
3. Try `SELECT` (should work if granted)
4. Try `INSERT`/`UPDATE` (should fail if not granted)

## Section 2 — dbutils.widgets (Quick Recap) 🟢 RUNS TODAY

Already used throughout this course for `catalog`/`schema` parameters — worth stating
explicitly once: widgets are visible, not protected. Never put anything sensitive in
a widget's default value.

In [0]:
dbutils.widgets.text("demo_region", "South", "Example widget — visible in the notebook UI")
print(f"Widget value: {dbutils.widgets.get('demo_region')}")
print("Anyone who can open this notebook can see and change this value directly in the UI.")

Widget value: South
Anyone who can open this notebook can see and change this value directly in the UI.


## Section 3 — dbutils.secrets 🟢 RUNS TODAY (mostly)

Secret **scope creation** requires the Databricks CLI or REST API — there's no
in-notebook command for it. **Reading** an existing secret with `dbutils.secrets.get`
works fine from any notebook cell, and Databricks automatically redacts the value
(`[REDACTED]`) anywhere it would otherwise print to cell output or logs.

**One-time, outside this notebook** (terminal, with the Databricks CLI configured):
```bash
databricks secrets create-scope --scope quickbite_secrets
databricks secrets put-secret --scope quickbite_secrets --key pg_password
# (prompts you to enter the value — it is never stored in your shell history)
```

In [0]:
try:
    pg_password = dbutils.secrets.get(scope="quickbite_secrets", key="pg_password")
    print("Retrieved successfully. The actual value never appears here:")
    print(pg_password)   # Databricks auto-redacts this to [REDACTED] in the cell output
except Exception as e:
    print("No scope named 'quickbite_secrets' exists yet in this workspace — expected")
    print("if you haven't run the CLI step above. The mechanism is what matters here:")
    print(f"{type(e).__name__}: {str(e)[:200]}")

No scope named 'quickbite_secrets' exists yet in this workspace — expected
if you haven't run the CLI step above. The mechanism is what matters here:
Py4JJavaError: An error occurred while calling GetSecret.
: java.lang.IllegalArgumentException: Secret does not exist with scope: quickbite_secrets and key: pg_password
	at com.databricks.backend.common.rpc.SimpleSe


### 🖱️ UI Instructions: Managing Secrets

**Important:** Secret scopes CANNOT be created via the UI. You must use the CLI or REST API.

**Using the Databricks CLI to create secrets:**

1. **Install the CLI:**
   ```bash
   pip install databricks-cli
   ```

2. **Configure authentication:**
   ```bash
   databricks configure --token
   ```
   * Host: Your workspace URL (e.g., `https://community.cloud.databricks.com`)
   * Token: Generate from **User Settings** → **Access Tokens** → **Generate New Token**

3. **Create a secret scope:**
   ```bash
   databricks secrets create-scope --scope quickbite_secrets
   ```

4. **Add a secret:**
   ```bash
   databricks secrets put-secret --scope quickbite_secrets --key pg_password
   ```
   * This opens an editor where you paste the secret value
   * The value is never stored in shell history

5. **List scopes and secrets:**
   ```bash
   databricks secrets list-scopes
   databricks secrets list-secrets --scope quickbite_secrets
   ```

**Note:** Secret values cannot be retrieved via CLI or UI once set (by design for security).

## Section 4 — Service Principals 🔴 CONCEPTUAL ONLY

Created and managed exclusively through the **account console** or Account API —
both explicitly unavailable on Free Edition. The syntax below is genuine and correct;
there's no way to execute the creation step here.

```text
# Account console (a paid/trial workspace, not Free Edition):
# Settings > Identity and access > Service principals > Add service principal
# -> gives you a Client ID and (once) a Client Secret
```
```sql
-- Grant it Unity Catalog privileges exactly like a human user, referencing its
-- Application (Client) ID:
GRANT USE CATALOG ON CATALOG quickbite_prod TO `a1b2c3d4-e5f6-...-application-id`;
GRANT SELECT ON TABLE quickbite_prod.gold.revenue_by_store TO `a1b2c3d4-...`;
```
```python
# A job or external script authenticates AS the service principal via OAuth
# (client credentials grant) — never with a human's personal access token:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient(
    host="https://<workspace>.cloud.databricks.com",
    client_id="a1b2c3d4-e5f6-...",
    client_secret=dbutils.secrets.get("quickbite_secrets", "sp_client_secret"),
)
```
**Why this matters in production:** a nightly job authenticating as a specific person
breaks the day that person changes their password, is offboarded, or is just on leave
and someone rotates credentials. A service principal's lifecycle is independent of any
one person's employment status.

### 🖱️ UI Instructions: Managing Service Principals

**Requirements:** Account console access (NOT available on Free Edition)

**To create a service principal (paid/trial workspaces only):**

1. **Access Account Console:**
   * Go to `https://accounts.cloud.databricks.com` (or your cloud's account URL)
   * Sign in with account admin credentials

2. **Create service principal:**
   * Click **Settings** in left sidebar
   * Select **Identity and access** → **Service principals**
   * Click **Add service principal**
   * Enter a name (e.g., `quickbite-etl-sp`)
   * Click **Add**

3. **Generate credentials:**
   * Click on the newly created service principal
   * Go to **OAuth secrets** tab
   * Click **Generate secret**
   * Copy the **Client ID** and **Client Secret** immediately (shown only once)
   * Store these in a secret manager or dbutils.secrets scope

4. **Assign to workspace:**
   * In the service principal details, click **Workspaces** tab
   * Click **Add access**
   * Select your workspace(s)
   * Click **Confirm**

5. **Grant UC permissions:**
   * Use SQL (as shown in cell above) or
   * Navigate to Catalog Explorer → select object → **Permissions** tab
   * Grant using the service principal's Application ID

## Section 8 — Delta Sharing: Databricks-to-Databricks (D2D) 🟡 PARTIAL

Being a Delta Sharing **recipient** is confirmed to work on Free Edition. Being a
**provider** (creating shares/recipients yourself) has multiple community reports of
hitting a permissions wall specific to Free Edition — so the code below is genuine,
correct syntax, written as though you have a full workspace to run the provider side
from, or a colleague's workspace willing to be the provider so your Free Edition
account can be a real, working recipient.

```sql
-- PROVIDER side (paid/trial workspace):
CREATE SHARE quickbite_partner_share
COMMENT 'QuickBite revenue data for partner analytics';

ALTER SHARE quickbite_partner_share ADD TABLE quickbite_prod.gold.revenue_by_store;

-- The recipient's own Databricks org gives you their sharing identifier for this:
CREATE RECIPIENT partner_analytics USING ID '<their-sharing-identifier>';

GRANT SELECT ON SHARE quickbite_partner_share TO RECIPIENT partner_analytics;
```

```sql
-- RECIPIENT side (this could genuinely be your Free Edition account):
SHOW SHARES IN PROVIDER quickbite_org;

CREATE CATALOG partner_view USING SHARE quickbite_org.quickbite_partner_share;

SELECT * FROM partner_view.gold.revenue_by_store;  -- reads live, no data copied
```
**What makes this "D2D":** the recipient authenticates as a genuine Databricks
identity in their own metastore — no credential file changes hands, and the shared
data shows up as an ordinary-looking UC catalog they can query ordinarily.

### 🖱️ UI Instructions: Delta Sharing (Databricks-to-Databricks)

**PROVIDER Side (creating and managing shares):**

1. **Create a Share:**
   * Navigate to **Catalog** in left sidebar
   * Click **Delta Sharing** tab (top of Catalog Explorer)
   * Click **Share data** button
   * Click **Create share**
   * Enter share name (e.g., `quickbite_partner_share`)
   * Add description
   * Click **Create**

2. **Add tables to the share:**
   * In the share details page, click **Add assets**
   * Select the catalog → schema → table(s) to share
   * Click **Add**
   * Can add multiple tables/views from different schemas

3. **Create a recipient (D2D):**
   * In the share details, click **Recipients** tab
   * Click **Add recipient**
   * Select **Databricks-to-Databricks sharing**
   * Enter recipient name
   * Get the **Sharing Identifier** from the recipient's workspace:
     - Recipient goes to **Catalog** → **Delta Sharing** → **Get sharing identifier**
   * Paste the identifier
   * Click **Create and grant access**

4. **Grant access:**
   * The recipient is automatically granted SELECT on the share
   * Can revoke access from Recipients tab

**RECIPIENT Side (accessing shared data):**

1. **View available shares:**
   * Navigate to **Catalog** → **Delta Sharing** tab
   * Click **New provider** to see shares from a provider

2. **Create catalog from share:**
   * Click on the available share
   * Click **Create catalog**
   * Enter catalog name (e.g., `partner_view`)
   * Click **Create**

3. **Query the data:**
   * The shared tables appear as a regular Unity Catalog
   * Browse to catalog → schema → table
   * Query normally: `SELECT * FROM partner_view.gold.revenue_by_store`
   * Data is read live from provider (no copying)

## Section 9 — Delta Sharing: Databricks-to-Open (D2O) 🟡 PARTIAL

The recipient needs **no Databricks account at all** here — a real advantage over D2D
for sharing outside organizations that use Databricks. Provider-side creation may hit
the same Free Edition restriction noted in Section 8; the recipient-side code below
is genuinely runnable in any Python environment once you have a credential file from
any provider (a colleague's workspace, or a public sample share if one is available —
search Databricks' current docs for an up-to-date sample, since these change).

```sql
-- PROVIDER side:
CREATE RECIPIENT open_partner USING BEARER TOKEN;
GRANT SELECT ON SHARE quickbite_partner_share TO RECIPIENT open_partner;
-- Databricks generates a config.share credential file + one-time activation link
-- to send the recipient — this is the ONLY thing that crosses the wire, never a
-- password or long-lived cloud credential.
```

### 🖱️ UI Instructions: Delta Sharing (Databricks-to-Open Protocol)

**PROVIDER Side (sharing with non-Databricks users):**

1. **Create the share** (same as D2D - see above)

2. **Create an open recipient:**
   * In the share details, go to **Recipients** tab
   * Click **Add recipient**
   * Select **Open sharing (non-Databricks)** or **Token-based recipient**
   * Enter recipient name
   * Click **Create**

3. **Generate credential file:**
   * Databricks generates a `config.share` file and activation link
   * Click **Download credential file**
   * Share this file securely with the recipient (email, encrypted channel)
   * The file contains:
     - Endpoint URL
     - Bearer token
     - Share name
   * **Important:** This is shown only once — download immediately

4. **Send to recipient:**
   * Share the `config.share` file
   * Provide the activation link (optional, for first-time setup)
   * Recipient needs NO Databricks account

**RECIPIENT Side (using the credential file):**

**Option 1: Python (any environment)**
```python
import delta_sharing
import pandas as pd

# Load the profile
profile_file = "path/to/config.share"
client = delta_sharing.SharingClient(profile_file)

# List available tables
tables = client.list_all_tables()
for table in tables:
    print(f"{table.share}.{table.schema}.{table.name}")

# Read data
table_url = f"{profile_file}#share_name.schema_name.table_name"
df = delta_sharing.load_as_pandas(table_url)
print(df.head())
```

**Option 2: Power BI, Tableau, etc.**
* Use the Delta Sharing connector
* Import the `config.share` file
* Connect and query the shared tables

**Key difference from D2D:** Recipient uses open-source `delta-sharing` library, no Databricks account needed

In [0]:
# RECIPIENT side — genuinely runnable in ANY Python environment (not just Databricks)
# once you have a real config.share file from a provider.
try:
    import delta_sharing

    profile_path = "config.share"   # the file the provider sends you
    client = delta_sharing.SharingClient(profile_path)
    print("Available tables:")
    for t in client.list_all_tables():
        print(f"  {t.share}.{t.schema}.{t.name}")

    table_url = f"{profile_path}#quickbite_partner_share.gold.revenue_by_store"
    df = delta_sharing.load_as_pandas(table_url)
    display(df)
except ImportError:
    print("pip install delta-sharing   # not installed in this environment")
except FileNotFoundError:
    print("No config.share file present — expected without a real provider credential.")
    print("The mechanism is the point: this exact code is what a recipient with zero")
    print("Databricks account of their own would run to read your shared data.")

pip install delta-sharing   # not installed in this environment


## Section 10 — Lakehouse Federation 🟡 PARTIAL

Queries an **external** database live, without copying its data into Delta first.
Needs a real, internet-reachable database to point at — a free-tier Postgres
instance (Neon, Supabase) takes a few minutes to set up if you don't have one handy.
The `CREATE CONNECTION`/`CREATE FOREIGN CATALOG` objects below are genuine Unity
Catalog objects — the same family as the storage credentials from Section 5.

In [0]:
dbutils.widgets.text("pg_host", "", "Postgres host (leave blank to skip execution)")
dbutils.widgets.text("pg_database", "quickbite_ext", "Postgres database name")
pg_host = dbutils.widgets.get("pg_host")
pg_database = dbutils.widgets.get("pg_database")

if pg_host:
    spark.sql(f"""
        CREATE CONNECTION IF NOT EXISTS postgres_conn TYPE postgresql
        OPTIONS (
          host '{pg_host}',
          port '5432',
          user secret('quickbite_secrets', 'pg_user'),
          password secret('quickbite_secrets', 'pg_password')
        )
    """)
    spark.sql(f"""
        CREATE FOREIGN CATALOG IF NOT EXISTS quickbite_postgres USING CONNECTION postgres_conn
        OPTIONS (database '{pg_database}')
    """)
    display(spark.sql("SHOW SCHEMAS IN quickbite_postgres"))
    print(f"Connected — browse quickbite_postgres.<schema>.<table> like any other UC object.")
else:
    print("pg_host widget is blank — showing the syntax without executing it.")
    print(f"""
CREATE CONNECTION postgres_conn TYPE postgresql
OPTIONS (
  host '<your-postgres-host>',
  port '5432',
  user secret('quickbite_secrets', 'pg_user'),
  password secret('quickbite_secrets', 'pg_password')
);

CREATE FOREIGN CATALOG quickbite_postgres USING CONNECTION postgres_conn
OPTIONS (database '{pg_database}');

SELECT * FROM quickbite_postgres.public.legacy_customer_master;
    """)

pg_host widget is blank — showing the syntax without executing it.

CREATE CONNECTION postgres_conn TYPE postgresql
OPTIONS (
  host '<your-postgres-host>',
  port '5432',
  user secret('quickbite_secrets', 'pg_user'),
  password secret('quickbite_secrets', 'pg_password')
);

CREATE FOREIGN CATALOG quickbite_postgres USING CONNECTION postgres_conn
OPTIONS (database 'quickbite_ext');

SELECT * FROM quickbite_postgres.public.legacy_customer_master;
    


### 🖱️ UI Instructions: Lakehouse Federation (External Database Connections)

**Prerequisites:**
* An external database (PostgreSQL, MySQL, SQL Server, Snowflake, etc.)
* Database credentials stored in a secret scope
* Internet-reachable database endpoint (or appropriate network configuration)

**Step 1: Create a Connection (via UI or SQL)**

**Via UI:**
1. **Navigate to Catalog:**
   * Click **Catalog** in left sidebar
   * Select **External Data** tab
   * Click **Create** → **Connection**

2. **Configure connection:**
   * **Name:** e.g., `postgres_conn`
   * **Type:** Select database type (PostgreSQL, MySQL, SQL Server, Snowflake, etc.)
   * **Host:** Database hostname/IP
   * **Port:** Database port (e.g., 5432 for PostgreSQL)
   * **Authentication:**
     - Select **Username and password**
     - Reference secrets: `secret('scope_name', 'username_key')` and `secret('scope_name', 'password_key')`
   * Click **Create**

**Via SQL (as shown in cell above):**
```sql
CREATE CONNECTION postgres_conn TYPE postgresql
OPTIONS (
  host 'your-database.example.com',
  port '5432',
  user secret('quickbite_secrets', 'pg_user'),
  password secret('quickbite_secrets', 'pg_password')
);
```

**Step 2: Create a Foreign Catalog**

**Via UI:**
1. **In Catalog Explorer:**
   * Click **Create** → **Foreign catalog**
   * **Name:** e.g., `quickbite_postgres`
   * **Connection:** Select the connection created above
   * **Database:** Name of the external database
   * Click **Create**

**Via SQL:**
```sql
CREATE FOREIGN CATALOG quickbite_postgres 
USING CONNECTION postgres_conn
OPTIONS (database 'quickbite_ext');
```

**Step 3: Query the External Data**

1. **Browse in Catalog Explorer:**
   * The foreign catalog appears alongside your Unity Catalog catalogs
   * Expand to see schemas and tables from the external database
   * Metadata is fetched live from the source

2. **Query normally:**
   ```sql
   -- Query external tables just like Unity Catalog tables
   SELECT * FROM quickbite_postgres.public.legacy_customer_master;
   
   -- Join external and UC tables
   SELECT 
       uc.customer_id,
       uc.full_name,
       ext.legacy_id
   FROM main.uc_governance_demo.customers uc
   JOIN quickbite_postgres.public.legacy_customer_master ext
     ON uc.customer_id = ext.new_customer_id;
   ```

3. **Performance tips:**
   * Predicates are pushed down to the external database when possible
   * Use `EXPLAIN` to verify pushdown
   * Consider creating Delta tables for frequently accessed external data

**Managing connections:**
* View connections: **Catalog** → **External Data** → **Connections** tab
* Edit/delete connections: Click the connection → **⋮** menu
* Grant usage: Only users with `USE CONNECTION` privilege can create foreign catalogs

## Recap

| Section | Status | What you ran | UI Instructions Added |
|---|---|---|---|
| ACLs & inheritance | 🟢 | `GRANT`/`SHOW GRANTS` (needs manual approval) | ✅ Catalog Explorer permissions |
| Second-user access test | 🔴 | Conceptual (needs 2nd user) | ✅ How to add users / use Delta Sharing |
| dbutils.widgets | 🟢 | ✅ Executed successfully | N/A (notebook feature) |
| dbutils.secrets | 🟢 | ✅ Executed (demo, no real scope) | ✅ CLI commands for scope creation |
| Service principals | 🔴 | Conceptual (needs account console) | ✅ Account console workflow |
| Delta Sharing D2D | 🟡 | Conceptual (provider needs paid tier) | ✅ Full provider & recipient UI flow |
| Delta Sharing D2O | 🟡 | Conceptual (no credential file) | ✅ Full provider & recipient workflow |
| Lakehouse Federation | 🟡 | ✅ Executed (no real DB configured) | ✅ Full connection setup & querying |



## 🎬 Production Demonstration Guide

### 📸 Screenshots to Capture (for presentations):

**1. Permissions (Catalog Explorer):**
* Navigate to [main.uc_governance_demo.customers](#table)
* Click **Permissions** tab
* Screenshot showing the grant list after running Cell 5
* Capture: principal names, privilege types (SELECT, USE CATALOG, etc.)

**2. Catalog Explorer - External Data:**
* Go to **Catalog** → **External Data** tab
* Screenshot showing Storage Credentials and Connections
* Shows the distinction from secret scopes (which don't appear here)

**3. Delta Sharing (if available):**
* **Catalog** → **Delta Sharing** tab
* Screenshot of shares list (or "Get sharing identifier" button)
* Shows the D2D recipient mechanism

**4. Retention Pipeline Results:**
* Cell 18 output: soft delete with timestamp
* Cell 19 output: hard delete confirmation (4 rows remain from 5)
* Shows the complete GDPR-compliant deletion workflow

**5. Workflows (for retention job):**
* Navigate to **Workflows**
* If you create the retention job (see UI instructions above)
* Screenshot the job configuration with schedule

---

### 👨‍🏫 What You Can Demonstrate in Production:

**✅ Runnable Today (Free Edition):**
1. **UC Permission Inheritance** — show how catalog-level grants cascade
2. **Secret Redaction** — demonstrate dbutils.secrets.get with [REDACTED]
3. **Retention Pipeline** — full 4-stage deletion workflow with audit trail
4. **UC vs Secret Scopes** — show SHOW STORAGE CREDENTIALS output
5. **Lakehouse Federation Syntax** — explain how to connect external DBs

**🟡 Demonstrable with Setup (requires external resources):**
1. **Second User Testing** — use Delta Sharing or trial workspace
2. **Delta Sharing D2D** — set up with a colleague's workspace
3. **Delta Sharing D2O** — share with a non-Databricks user
4. **Lakehouse Federation** — connect to a real Postgres/MySQL instance (free tier: Neon, Supabase, PlanetScale)

**📝 Reference-Only (needs paid workspace/account console):**
1. **Service Principals** — show the syntax and explain the workflow
2. **Compute Access Modes** — explain Shared vs Dedicated isolation

---

### 🛠️ Quick Setup for Full Demo:

**If you want to run EVERYTHING:**

1. **Get a trial workspace** (14 days free):
   * Sign up at databricks.com/try-databricks
   * Get account console access for service principals
   * Add a second user for permissions testing

2. **Set up external database** (5 minutes):
   * Create free Postgres on [Neon](https://neon.tech) or [Supabase](https://supabase.com)
   * Create a test table
   * Update the `pg_host` widget in Cell 27
   * Create secret scope with credentials (CLI commands in UI instructions above)

3. **Test Delta Sharing with a colleague:**
   * Have them share a table with you (D2D recipient)
   * Or create a share and send them a config.share file (D2O)

**Result:** Every section will be fully executable, with real multi-user testing and live external data queries.

---

### 🎯 Key Takeaways for Your Presentation:

✅ **Unity Catalog ACLs** use SQL GRANTs that cascade from catalog → schema → table  
✅ **Secrets** (dbutils.secrets) are workspace-level, separate from UC credentials  
✅ **Service principals** enable automation without depending on individual users  
✅ **Retention pipelines** handle GDPR/compliance with soft delete → grace period → hard delete  
✅ **Compute isolation** (Shared vs Dedicated) affects UC enforcement and API support  
✅ **Delta Sharing** enables live data sharing without copying (D2D or D2O)  
✅ **Lakehouse Federation** queries external databases in place (no ETL required)  

**All code is production-ready** — this notebook is a template you can adapt for real use cases!